In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import dask.dataframe as dd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import scipy.stats
from scipy.stats import chi2
%matplotlib inline
import xgboost as xgb
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import CategoricalNB
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.decomposition import PCA
from mlxtend.plotting import plot_decision_regions


%matplotlib inline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier


Train and Test data scaling


In [4]:
import dask.array as da
df = pd.read_csv("/content/drive/MyDrive/new_train2.csv")

In [5]:
df_test = pd.read_csv("/content/drive/MyDrive/new_test2.csv")

In [6]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df_test = df_test.loc[:, ~df_test.columns.str.contains('^Unnamed')]

In [7]:
df.head()

,TotLen Fwd Pkts,SYN Flag Cnt,Fwd Seg Size Min,Bwd IAT Mean,Pkt Len Min,Flow Byts/s,Bwd Pkts/s,Init Fwd Win Byts,CWE Flag Count,Label
0,49.0,0.0,8.0,0.000000,49.0,92972.501091,436.490615,-1.0,0.0,0
1,0.0,0.0,20.0,0.000000,0.0,0.000000,0.172838,8192.0,0.0,0
2,287.0,0.0,20.0,4502.333333,0.0,90431.436390,296.011248,65535.0,0.0,1
3,308.0,0.0,20.0,535.333333,0.0,772049.689400,2484.472050,65535.0,0.0,1
4,36.0,0.0,8.0,0.000000,36.0,97757.847534,896.860987,-1.0,0.0,0


In [8]:
df_test.head()

,TotLen Fwd Pkts,SYN Flag Cnt,Fwd Seg Size Min,Bwd IAT Mean,Pkt Len Min,Flow Byts/s,Bwd Pkts/s,Init Fwd Win Byts,CWE Flag Count
0,77.0,1.0,20.0,0.0,0.0,46581.972172,0.000000,255.0,0.0
1,77.0,1.0,20.0,0.0,0.0,107994.389902,0.000000,258.0,0.0
2,740.0,0.0,20.0,3429443.5,0.0,215.040927,0.339411,8192.0,0.0
3,33.0,0.0,8.0,0.0,33.0,90840.272521,757.002271,-1.0,0.0
4,47.0,0.0,8.0,0.0,47.0,214765.100671,1342.281879,-1.0,0.0


In [9]:
y=df["Label"]
df = df.drop("Label",axis=1)
X=df

In [10]:
y.value_counts()

0    7610849
1     782231
Name: Label, dtype: int64

In [11]:
from sklearn.preprocessing import StandardScaler
std = StandardScaler()
X_t = std.fit_transform(X)
df_t = std.transform(df_test)

XGBOOST CLASSIFICATION

In [12]:
xgb_clf= xgb.XGBClassifier(
                        n_estimators=10,
                        max_depth=6,
                        objective="binary:logistic",
                        learning_rate=.3,
                        subsample=1,
                        scale_pos_weight=9.72,
                        min_child_weight=2,
                        colsample_bytree=1,
                        tree_method='hist',
                        use_label_encoder=False
                        , eta=0.05
                        )


xgb_clf.fit(X_t,y)


XGBClassifier(eta=0.05, learning_rate=0.3, max_depth=6, min_child_weight=2,
              n_estimators=10, scale_pos_weight=9.72, tree_method='hist',
              use_label_encoder=False)

In [13]:
y_pred_xgb = xgb_clf.predict(df_t)

In [14]:
predd_rfb = pd.DataFrame(y_pred_xgb, columns =["Label"])
predd_rfb.to_csv('Prediction_final.csv', index=True)  

LOGISTIC REGRESSION

In [15]:

from sklearn.linear_model import LogisticRegression
lr2 = LogisticRegression(solver='lbfgs', max_iter=1000000, n_jobs=-1, class_weight='balanced')

In [16]:
lr2.fit(X_t, y)

LogisticRegression(class_weight='balanced', max_iter=1000000, n_jobs=-1)

STACKED CLASSIFIER

In [17]:
model_names = ["lr2","xgb_clf"]

In [18]:
model_vars = [eval(n) for n in model_names]
model_list = list(zip(model_names, model_vars))

In [19]:

stacked = StackingClassifier(estimators=model_list)

stacked.fit(X_t, y)


StackingClassifier(estimators=[('lr2',
                                LogisticRegression(class_weight='balanced',
                                                   max_iter=1000000,
                                                   n_jobs=-1)),
                               ('xgb_clf',
                                XGBClassifier(eta=0.05, learning_rate=0.3,
                                              max_depth=6, min_child_weight=2,
                                              n_estimators=10,
                                              scale_pos_weight=9.72,
                                              tree_method='hist',
                                              use_label_encoder=False))])

In [20]:
y_preds = stacked.predict(df_t)

In [21]:
predd_rfb = pd.DataFrame(y_preds, columns =["Label"])
predd_rfb.to_csv('Prediction_stack_final.csv', index=True)  